### Scenario Site

### 작가지망생의 미로 찾기

In [ ]:
pip install selenium webdriver-manager

In [ ]:
links = []
bullten_boards = []
link_type = {'HmM3/':'2~77', 'M3Nt/':'1~646', 'ObVe/':'1~5', 'K84u/':'1~178',
             '8E0K/':'29~1735', '8Dzy/':'24~2136', '8DzI/':'1~1543', 'GxmD/':'1~218',
             'S58G/':'1~8', 'IXuT/':'1~283', 'K3Ri': '24~265'}

link_bullentin_board = {'HmM3/':'시나리오[국외]', 'M3Nt/':'시나리오[국내]', 'ObVe/':'외드대본',
                        'K84u/':'타방송대본', '8E0K/':'SBS대본', '8Dzy/':'MBC대본', 
                        '8DzI/':'KBS대본', 'GxmD/':'단막극대본', 'S58G/':'오늘의 대본', 
                        'IXuT/':'시놉시스', 'K3Ri': '미완결대본'}

for (key, val), (key2, val2) in zip(link_type.items(), link_bullentin_board.items()):
    val_split = val.split("~")
    start = int(val_split[0])
    end = int(val_split[1])
    for num in range(start, end + 1):
        link = 'https://cafe.daum.net/ygy2317/' + key + str(num)
        links.append(link)
        bullten_boards.append(val2)

In [21]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Setup Chrome options (headless means the browser runs in the background)
chrome_options = Options()
chrome_options.add_argument("--headless") 

driver = webdriver.Chrome(options=chrome_options)

try:
    url = 'https://cafe.daum.net/ygy2317/K3Ri/265'
    driver.get(url)
    
    # Wait for the iframe to load
    time.sleep(3) 
    
    # Daum Cafe posts are usually inside an iframe with id='down'
    driver.switch_to.frame('down')
    
    # Extract the title which acts as the 'pdf_file_name' during Ctrl+P
    # Usually, this is the text within the .item_subject or the document title
    raw_title = driver.title
    
    # Sanitize the title to make it a valid filename
    pdf_file_name = "".join([c for c in raw_title if c.isalnum() or c in (' ', '.', '_')]).strip()
    pdf_file_name = pdf_file_name.replace("작가지망생의 미로찾기", "").replace("Daum 카페", "")
    pdf_file_name = pdf_file_name.strip()    
    # pdf_file_name += ".pdf"

    print(f"Extracted PDF Filename: {pdf_file_name}")

finally:
    driver.quit()

Extracted PDF Filename: 미스터 션샤인 14


In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# 1. Setup Chrome Options
chrome_options = Options()
chrome_options.add_argument("--headless") 
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

links = []
link_bullentin_boards = []
pdf_file_names = []

# Configuration
link_type = {
    'HmM3/':'2~77', 'M3Nt/':'1~646', 'ObVe/':'1~5', 'K84u/':'1~178',
    '8E0K/':'29~1735', '8Dzy/':'24~2136', '8DzI/':'1~1543', 'GxmD/':'1~218',
    'S58G/':'1~8', 'IXuT/':'1~283', 'K3Ri/': '24~265' # Added / to K3Ri for consistency
}

link_bullentin_board = {
    'HmM3/':'시나리오[국외]', 'M3Nt/':'시나리오[국내]', 'ObVe/':'외드대본',
    'K84u/':'타방송대본', '8E0K/':'SBS대본', '8Dzy/':'MBC대본', 
    '8DzI/':'KBS대본', 'GxmD/':'단막극대본', 'S58G/':'오늘의 대본', 
    'IXuT/':'시놉시스', 'K3Ri/': '미완결대본'
}

try:
    for key in link_type.keys():
        val = link_type[key]
        board_name = link_bullentin_board[key]
        
        val_split = val.split("~")
        start = int(val_split[0])
        end = int(val_split[1])
        
        for num in range(start, end + 1):
            link = f'https://cafe.daum.net/ygy2317/{key}{num}'
            # print(link)
            
            try:
                driver.get(link)
                time.sleep(2) # Give it a moment to load
                
                # Switch to the content iframe
                driver.switch_to.frame('down')
                
                # Get the title
                raw_title = driver.title
                
                # Clean the title
                clean_name = raw_title.replace("작가지망생의 미로찾기", "").replace("Daum 카페", "").strip()
                # Remove special characters for filename safety
                pdf_file_name = "".join([c for c in clean_name if c.isalnum() or c in (' ', '.', '_')]).strip()
                
                # print(f"Extracted PDF Filename: {pdf_file_name}")
                
                pdf_file_names.append(pdf_file_name)
                links.append(link)
                link_bullentin_boards.append(board_name)
                
            except Exception as e:
                print(f"Error processing {link}: {e}")
            finally:
                # CRITICAL: Switch back to main content for the next iteration
                driver.switch_to.default_content()

finally:
    # Close the browser only ONCE at the very end
    driver.quit()

# Save to Excel
df = pd.DataFrame({'Bulletin': link_bullentin_boards, 'Post': pdf_file_names, 'Link': links})
df.to_excel('drama_list.xlsx', index=False)
print("File saved successfully.")

File saved successfully.


In [33]:
import re


df = pd.read_excel('drama_list.xlsx')
posts = list(df['Post'])

cleaned_posts = [item for item in posts if not isinstance(item, float)]
cleaned_posts = [re.sub(r'[^가-힣]', '', item) for item in cleaned_posts]
cleaned_posts = list(dict.fromkeys(cleaned_posts))

"""
# 1. Identify common prefixes by comparing strings
found_titles = set()

for item_a in cleaned_posts:
    for item_b in cleaned_posts:
        if item_a == item_b:
            continue
        
        # Find the shared starting characters between two different strings
        common_prefix = ""
        for char_a, char_b in zip(item_a, item_b):
            if char_a == char_b:
                common_prefix += char_a
            else:
                break
        
        # If a significant common prefix is found, add it to our set
        if common_prefix:
            found_titles.add(common_prefix)

# 2. Filter the set to keep only the shortest unique "root" titles
# (This removes the 'repeated' overlap and keeps the base title)
final_set = set()
sorted_titles = sorted(list(found_titles), key=len)

for title in sorted_titles:
    if not any(title.startswith(existing) for existing in final_set if existing != title):
        if len(title) > 1: # Ensure we don't catch single accidental char matches
            final_set.add(title)

# 3. Convert to list and maintain a clean output
cleaned_posts = sorted(list(final_set), key=lambda x: cleaned_posts.index(next(s for s in cleaned_posts if s.startswith(x))))

cleaned_posts = list(set(cleaned_posts))
"""
print(cleaned_posts)

['캐스트어웨이', '해리가샐리를만났을때', '하울의움직이는성', '가위손', '러브액츄얼리', '뮬란', '메멘토', '미녀와야수', '노팅힐', '러브레터', '미스터빈', '로마의휴일', '로빈훗', '매디슨카운티의다리', '라이언일병구하기', '맨인블랙', '글래디에이터', '드라큘라', '다이하드', '타이타닉', '덤앤더머', '그린마일', '슬리피할로우', '셰익스피어인러브', '식스센스', '뱀파이어와의인터뷰', '쇼생크탈출', '아마데우스', '피아노', '세븐', '포레스트검프', '매트릭스', '큐브', '죽은시인의사회', '페이스오프', '유주얼서스펙트', '트와일라잇', '월이야기', '시나리오목록', '다크나이트', '악마는프라다를입는다', '시간을달리는소녀', '다크나이트라이즈', '블랙스완', '반지의제왕반지원정대', '반지의제왕두개의탑', '프리퀀시', '빅', '굿윌헌팅', '마이걸', '아이즈와이드셧', '나비효과', '버드맨', '캐롤', '인터스텔라', '위플래쉬', '인사이드아웃', '인셉션', '허', '그래비티', '보이후드', '이보다더좋을순없다', '나를찾아줘', '스포트라이트', '겨울왕국', '실버라이닝플레이북', '나이트크롤러', '리플리', '빅피쉬', '소셜네트워크', '인턴', '브로크백마운틴', '차이나타운', '사랑도통역이되나요', '엽기적인그녀곽재용', '공공의적', '올드보이박찬욱', '연애소설이한', '국화꽃향기김희재이정욱', '해안선김기덕', '서편제', '지구를지켜라', '달콤한인생', '공동경비구역', '동감', '해피엔드', '인어공주', '오로라공주', '박수칠때떠나라장진', '왕의남자', '파랑주의보', '광식이동생광태김현석', '와일드카드이만희', '살인의추억봉준호심성보', '번지점프를하다고은님', '실미도김희재', '범죄의재구성최동훈', '장화홍련', '애정결핍이두남자에게미치는영향', '우리들의일그러진영웅', '은행나무침대강제규', '추격자나홍진', '사랑니정지우', '김씨표류기이